# Rozdział 2 — Od sygnału ciągłego do próbek

Doświadczenia E2.1–E2.7 odtwarzają figury i wyniki rozdziału.
Wykorzystują modele analityczne oraz dołączony fragment zapisu EKG.
Obliczenia nie wymagają pobierania danych.

Rewizja D01 (12 listopada 2026): położenia adnotacji EKG pochodzą z oryginalnych plików WFDB `.atr`, v1.0.0. Próbki sygnału pozostają zgodne z wycinkami CPSB. Ślad korekty: `chapter_notes/korekta_adnotacji_D01.md`.


In [1]:
import csv
import json
from pathlib import Path
import platform
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import scipy
from scipy.signal import butter, freqs

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "data/ch01_manifest.json").is_file()
            and (p / "notebooks/ch02_sampling_aliasing.ipynb").is_file())
sys.path.insert(0, str(root / "src"))
from bspsbook.measurement import file_digest, load_ecg

figures, outputs = root / "figures/generated", root / "results"
figures.mkdir(parents=True, exist_ok=True)
outputs.mkdir(exist_ok=True)
plt.rcParams.update({
    "font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": .20, "legend.frameon": False,
    "svg.hashsalt": "CPSB-ch02",
})
blue, orange, gray, green = "#095b89", "#a44b24", "#57646c", "#237866"
results = {}

def save(fig, stem):
    fig.savefig(figures / f"{stem}.png", dpi=175, metadata={"Software": "CPSB"})
    fig.savefig(figures / f"{stem}.svg", metadata={"Date": None})
    plt.close(fig)

def write_rows(name, rows):
    with (outputs / name).open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=tuple(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def tone(f, t):
    return np.cos(2 * np.pi * f * t)

## E2.1 — Siatka czasu i przypadek graniczny

In [2]:
fs = 200
ts = np.arange(13) / fs
td = np.linspace(0, ts[-1], 1201)
fig, axes = plt.subplots(3, 1, sharex=True, figsize=(8.3, 6.3), layout="constrained")
for ax, f in zip(axes[:2], (40, 96)):
    ax.plot(td * 1000, tone(f, td), color=gray, lw=1, label="Model ciągły")
    ax.scatter(ts * 1000, tone(f, ts), color=blue, s=19, zorder=3, label="Próbki")
    density = f"{fs/f:.2f}".replace(".", ",")
    ax.set(title=f"{f} Hz · {density} próbki na okres", ylabel="Amplituda [j.u.]")
axes[0].legend(loc="upper right", ncol=2, fontsize=8.5)
edge_sine = np.sin(2 * np.pi * 100 * ts)
edge_cos = tone(100, ts)
np.testing.assert_allclose(edge_sine, 0, atol=1e-13)
np.testing.assert_allclose(edge_cos, (-1.) ** np.arange(len(ts)), atol=1e-13)
axes[2].plot(td * 1000, np.sin(2 * np.pi * 100 * td), color=orange, lw=1, label="Sinus 100 Hz")
axes[2].plot(td * 1000, tone(100, td), color=blue, lw=1, ls="--", label="Cosinus 100 Hz")
axes[2].scatter(ts * 1000, edge_sine, color=orange, s=17, zorder=3)
axes[2].scatter(ts * 1000, edge_cos, color=blue, s=17, zorder=3)
axes[2].set(title="100 Hz · próbki zależą od fazy początkowej",
            ylabel="Amplituda [j.u.]", xlabel="Czas [ms]", xlim=(0, 60))
for ax in axes:
    ax.set_ylim(-1.10, 1.53)
axes[2].legend(loc="upper right", ncol=2, fontsize=8.5)
save(fig, "ch02_01_siatka")
results["E2.1"] = {"fs_hz": fs, "Ts_ms": 1000 / fs, "nyquist_hz": fs / 2,
                   "samples_per_cycle_40": fs / 40, "samples_per_cycle_96": fs / 96,
                   "edge_sine_max_abs": float(np.max(np.abs(edge_sine)))}
print(results["E2.1"])

{'fs_hz': 200, 'Ts_ms': 5.0, 'nyquist_hz': 100.0, 'samples_per_cycle_40': 5.0, 'samples_per_cycle_96': 2.0833333333333335, 'edge_sine_max_abs': 2.6954609193973537e-15}


## E2.2 — Dwa różne przebiegi dają te same próbki

In [3]:
a, b = 80, 120
n_long = np.arange(-300, 301)
x_low, x_high = tone(a, n_long / fs), tone(b, n_long / fs)
np.testing.assert_allclose(x_low, x_high, rtol=0, atol=5e-13)
t_plot = np.linspace(0, .060, 1201)
reconstructed = np.sinc(t_plot[:, None] * fs - n_long) @ x_high
error_reconstruction = float(np.max(np.abs(reconstructed - tone(a, t_plot))))
assert error_reconstruction < .004
show = np.arange(13) / fs
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8.3, 4.85), layout="constrained")
axes[0].plot(t_plot * 1000, tone(a, t_plot), color=blue, lw=1.25, label="Model 80 Hz")
axes[0].plot(t_plot * 1000, tone(b, t_plot), color=orange, lw=1, ls="--", label="Model 120 Hz")
axes[0].scatter(show * 1000, tone(b, show), color="#242c33", s=19, zorder=3, label="Wspólne próbki")
axes[0].legend(loc="upper right", ncol=3, fontsize=8.5)
axes[1].plot(t_plot * 1000, tone(b, t_plot), color=orange, lw=.9, ls="--", label="Wejście 120 Hz")
axes[1].plot(t_plot * 1000, reconstructed, color=blue, lw=1.15, label="Interpolacja z próbek")
axes[1].scatter(show * 1000, tone(b, show), color="#242c33", s=17, zorder=3)
axes[1].legend(loc="upper right", ncol=2, fontsize=8.5)
for ax in axes:
    ax.set(ylabel="Amplituda [j.u.]", ylim=(-1.10, 1.53))
axes[1].set(xlabel="Czas [ms]", xlim=(0, 60))
save(fig, "ch02_02_alias")
results["E2.2"] = {"input_hz": b, "alias_hz": a, "interpolator_samples": len(n_long),
                   "max_sample_difference": float(np.max(np.abs(x_low - x_high))),
                   "max_interpolation_error_vs_80": error_reconstruction}
print(results["E2.2"])

{'input_hz': 120, 'alias_hz': 80, 'interpolator_samples': 601, 'max_sample_difference': 3.765876499528531e-13, 'max_interpolation_error_vs_80': 3.9324160345022285e-05}


## E2.3 — Liczba próbek, czas obserwacji i krok siatki

In [4]:
timing_rows = []
for n, rate in ((200, 200), (200, 400), (800, 200)):
    t = np.arange(n) / rate
    grid = np.fft.rfftfreq(n, d=1 / rate)
    np.testing.assert_allclose(grid[1], 1 / (n / rate))
    timing_rows.append({"N": n, "fs_hz": rate, "sample_interval_ms": 1000 / rate,
                        "T_s": n / rate, "first_to_last_s": float(t[-1] - t[0]),
                        "DFT_step_hz": float(grid[1])})
write_rows("ch02_time_grid.csv", timing_rows)
results["E2.3"] = timing_rows
for row in timing_rows:
    print(row)

{'N': 200, 'fs_hz': 200, 'sample_interval_ms': 5.0, 'T_s': 1.0, 'first_to_last_s': 0.995, 'DFT_step_hz': 1.0}
{'N': 200, 'fs_hz': 400, 'sample_interval_ms': 2.5, 'T_s': 0.5, 'first_to_last_s': 0.4975, 'DFT_step_hz': 2.0}
{'N': 800, 'fs_hz': 200, 'sample_interval_ms': 5.0, 'T_s': 4.0, 'first_to_last_s': 3.995, 'DFT_step_hz': 0.25}


## E2.4 — Tłumienie przed próbkowaniem

In [5]:
rate, f_interest, f_noise = 80, 12, 68
A_interest, A_noise = .6, .4
order, cutoff = 6, 24
coeff_b, coeff_a = butter(order, 2 * np.pi * cutoff, analog=True)
_, H = freqs(coeff_b, coeff_a, 2 * np.pi * np.array([f_interest, f_noise]))
n = np.arange(2 * rate)
t_obs = n / rate
t_dense = np.linspace(0, .22, 1201)

def raw(t):
    return A_interest * tone(f_interest, t) + A_noise * tone(f_noise, t)

def useful_filtered(t):
    return np.real(A_interest * H[0] * np.exp(2j * np.pi * f_interest * t))

def filtered(t):
    return useful_filtered(t) + np.real(A_noise * H[1] * np.exp(2j * np.pi * f_noise * t))

def amplitude_at_12(y):
    return float(2 * abs(np.sum(y * np.exp(-2j * np.pi * f_interest * t_obs))) / len(y))

np.testing.assert_allclose(tone(f_noise, t_obs), tone(f_interest, t_obs), atol=2e-13)
assert np.isclose(amplitude_at_12(raw(t_obs)), 1.0)
residual_mV = float(A_noise * abs(H[1]))
amp_filtered = amplitude_at_12(filtered(t_obs))
assert abs(amp_filtered - A_interest * abs(H[0])) <= residual_mV
amp_sample_times = t_obs[t_obs <= t_dense[-1]]

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8.3, 5.35), layout="constrained")
axes[0].plot(t_dense * 1000, raw(t_dense), color=gray, lw=1, label="Model wejścia 12 + 68 Hz")
axes[0].plot(t_dense * 1000, A_interest * tone(f_interest, t_dense),
             color=blue, lw=.9, ls="--", label="Składowa 12 Hz na wejściu")
axes[0].scatter(amp_sample_times * 1000, raw(amp_sample_times), color=orange, s=23,
                label="Próbki bez filtru", zorder=3)
axes[1].plot(t_dense * 1000, useful_filtered(t_dense), color=blue, lw=1,
             label="Składowa 12 Hz po filtrze")
axes[1].scatter(amp_sample_times * 1000, filtered(amp_sample_times), color=orange, s=23,
                label="Próbki po filtrze", zorder=3)
for ax in axes:
    ax.set(ylabel="Napięcie [mV]", ylim=(-1.08, 1.46))
    ax.legend(loc="upper right", ncol=3, fontsize=8.2)
axes[1].set(xlabel="Czas [ms]", xlim=(0, 220))
save(fig, "ch02_03_antyalias")
results["E2.4"] = {
    "fs_hz": rate, "desired_hz": f_interest, "interference_hz": f_noise,
    "amp_before_mV": amplitude_at_12(raw(t_obs)), "amp_after_mV": amp_filtered,
    "amp_filtered_reference_mV": float(A_interest * abs(H[0])),
    "residual_alias_uV": 1000 * residual_mV,
    "attenuation_at_68_dB": float(-20 * np.log10(abs(H[1]))),
    "analog_filter_order": order, "analog_cutoff_hz": cutoff,
}
print(results["E2.4"])

{'fs_hz': 80, 'desired_hz': 12, 'interference_hz': 68, 'amp_before_mV': 1.0, 'amp_after_mV': 0.5992946346306981, 'amp_filtered_reference_mV': 0.5999267712208172, 'residual_alias_uV': 0.7731667235023664, 'attenuation_at_68_dB': 54.27573674535475, 'analog_filter_order': 6, 'analog_cutoff_hz': 24}


## E2.5 — Kwantyzacja i budżet amplitudowy

In [6]:
def quantize(values, bits, lower=-1., upper=1.):
    count = 2 ** bits
    delta = (upper - lower) / count
    in_range = (values >= lower) & (values < upper)
    codes = np.clip(np.floor((values - lower) / delta), 0, count - 1).astype(int)
    return codes, lower + (codes + .5) * delta, in_range, delta

t_quant = np.arange(800) / 400
slow = .05 + .02 * np.sin(2 * np.pi * 5 * t_quant)
wide = .9 * np.sin(2 * np.pi * 17.3 * t_quant)
rows = []
traces = {}
for bits in (4, 8, 12):
    codes, q, in_range, delta = quantize(slow, bits)
    _, q_wide, in_range_wide, _ = quantize(wide, bits)
    assert in_range.all() and in_range_wide.all()
    assert np.max(np.abs(q - slow)) <= delta / 2 + 1e-15
    assert np.max(np.abs(q_wide - wide)) <= delta / 2 + 1e-15
    rms = float(np.sqrt(np.mean((q_wide - wide) ** 2)))
    traces[bits] = q
    rows.append({
        "bits": bits, "LSB_uV": 1000 * delta, "codes_for_weak_tone": len(np.unique(codes)),
        "rms_error_uV_for_wide_tone": 1000 * rms, "uniform_error_RMS_uV": float(1000 * delta / np.sqrt(12)),
        "required_dB_for_residual_leq_1_LSB": float(20 * np.log10(A_noise / delta)),
    })
assert rows[0]["codes_for_weak_tone"] == 1 and rows[1]["codes_for_weak_tone"] == 6
assert rows[2]["LSB_uV"] < results["E2.4"]["residual_alias_uV"] < rows[1]["LSB_uV"]
write_rows("ch02_quantization.csv", rows)
results["E2.5"] = rows
for row in rows:
    print(row)

use = t_quant <= .42
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8.3, 4.95), layout="constrained")
for ax, bits in zip(axes, (4, 8)):
    ax.plot(t_quant[use] * 1000, slow[use] * 1000, color=blue, lw=1.2, label="Próbki wejścia")
    ax.step(t_quant[use] * 1000, traces[bits][use] * 1000, where="post", color=orange,
            lw=1.1, ls="--", label="Wartości po kwantyzacji")
    ax.set(title=f"{bits} bity" if bits == 4 else "8 bitów", ylabel="Napięcie [µV]", ylim=(25, 87))
axes[0].legend(loc="upper right", ncol=2, fontsize=8.5)
axes[1].set(xlabel="Czas [ms]", xlim=(0, 420))
save(fig, "ch02_04_kwantyzacja")

{'bits': 4, 'LSB_uV': 125.0, 'codes_for_weak_tone': 1, 'rms_error_uV_for_wide_tone': 38.1851362668375, 'uniform_error_RMS_uV': 36.08439182435161, 'required_dB_for_residual_leq_1_LSB': 10.102999566398122}
{'bits': 8, 'LSB_uV': 7.8125, 'codes_for_weak_tone': 6, 'rms_error_uV_for_wide_tone': 2.295550748556001, 'uniform_error_RMS_uV': 2.255274489021976, 'required_dB_for_residual_leq_1_LSB': 34.18539921951662}
{'bits': 12, 'LSB_uV': 0.48828125, 'codes_for_weak_tone': 37, 'rms_error_uV_for_wide_tone': 0.1409458260556643, 'uniform_error_RMS_uV': 0.1409546555638735, 'required_dB_for_residual_leq_1_LSB': 58.26779887263511}


## E2.6 — Szczyt w rzeczywistym zapisie EKG a początek siatki

In [7]:
record, manifest = load_ecg(root / "data")
ref_rate = manifest["fs_hz"]
voltage = record["signal_mV"]
seconds = record["time_s"]
selected = record["r_samples"][(seconds[record["r_samples"]] >= 12) & (seconds[record["r_samples"]] < 18)]
divisor = 3
window_radius = round(.036 * ref_rate)
peak_rows = []
for r in selected:
    idx = np.arange(r - window_radius, r + window_radius + 1)
    ref_max = float(np.max(voltage[idx]))
    for phase in range(divisor):
        n_phase = idx[idx % divisor == phase]
        index = int(n_phase[np.argmax(voltage[n_phase])])
        value = float(voltage[index])
        peak_rows.append({
            "record_annotation_time_s": float(seconds[r] + manifest["record_start_sample"] / ref_rate),
            "reference_sample_local": int(r), "phase_of_120_hz_grid": phase,
            "max_mV_360_hz": ref_max, "max_mV_120_hz": value,
            "difference_mV": ref_max - value, "time_of_120_hz_max_ms_relative": float((index-r) / ref_rate * 1000),
        })
assert len(peak_rows) == 21 and all(r["difference_mV"] >= 0 for r in peak_rows)
write_rows("ch02_ecg_grid.csv", peak_rows)
largest = max(peak_rows, key=lambda r: r["difference_mV"])
r = largest["reference_sample_local"]
chosen_rows = [row for row in peak_rows if row["reference_sample_local"] == r]
idx = np.arange(r - window_radius, r + window_radius + 1)
fig, axes = plt.subplots(3, 1, sharex=True, sharey=True, figsize=(8.3, 6.2), layout="constrained")
for phase, (ax, row) in enumerate(zip(axes, chosen_rows)):
    observed = idx[idx % divisor == phase]
    ax.plot((idx - r) / ref_rate * 1000, voltage[idx], ".-", color=gray, lw=.85, ms=3,
            label="Wartości w zapisie 360 Hz")
    ax.scatter((observed - r) / ref_rate * 1000, voltage[observed], color=blue, s=22,
               label="Wybrane próbki 120 Hz", zorder=3)
    ax.scatter(row["time_of_120_hz_max_ms_relative"], row["max_mV_120_hz"],
               facecolors="none", edgecolors=orange, s=90, zorder=4)
    ax.axvline(0, color=gray, lw=.75, ls=":")
    maximum = f"{row['max_mV_120_hz']:.3f}".replace(".", ",")
    ax.set(title=f"Siatka {phase} · maksimum {maximum} mV",
           ylabel="EKG [mV]", ylim=(-.65, 1.01))
axes[0].legend(loc="upper right", ncol=2, fontsize=8.2)
axes[-1].set(xlabel="Czas względem adnotacji [ms]")
save(fig, "ch02_05_ekg")
results["E2.6"] = {"annotations": len(selected), "window_radius_samples": window_radius,
                   "new_fs_hz": ref_rate / divisor, "largest_difference": largest,
                   "phases_at_selected_event": chosen_rows}
print(results["E2.6"])

{'annotations': 7, 'window_radius_samples': 13, 'new_fs_hz': 120.0, 'largest_difference': {'record_annotation_time_s': 644.2027777777778, 'reference_sample_local': 5113, 'phase_of_120_hz_grid': 0, 'max_mV_360_hz': 0.7400000095367432, 'max_mV_120_hz': 0.5950000286102295, 'difference_mV': 0.14499998092651367, 'time_of_120_hz_max_ms_relative': -2.7777777777777777}, 'phases_at_selected_event': [{'record_annotation_time_s': 644.2027777777778, 'reference_sample_local': 5113, 'phase_of_120_hz_grid': 0, 'max_mV_360_hz': 0.7400000095367432, 'max_mV_120_hz': 0.5950000286102295, 'difference_mV': 0.14499998092651367, 'time_of_120_hz_max_ms_relative': -2.7777777777777777}, {'record_annotation_time_s': 644.2027777777778, 'reference_sample_local': 5113, 'phase_of_120_hz_grid': 1, 'max_mV_360_hz': 0.7400000095367432, 'max_mV_120_hz': 0.7300000190734863, 'difference_mV': 0.009999990463256836, 'time_of_120_hz_max_ms_relative': 0.0}, {'record_annotation_time_s': 644.2027777777778, 'reference_sample_local

## E2.7 — Zestawienie konfiguracji i ślad wykonania

In [8]:
# Wiersze z tabeli 4 ADS1299-x (SBAS499C), zasilanie 5 V, VREF 4,5 V, PGA 24.
ti_rows = [
    {"output_sps": 250, "bandwidth_3dB_Hz": 65, "input_RMS_noise_uV": .14, "ENOB_as_reported": 19.85},
    {"output_sps": 1000, "bandwidth_3dB_Hz": 262, "input_RMS_noise_uV": .28, "ENOB_as_reported": 18.85},
]
write_rows("ch02_ads1299.csv", ti_rows)
results["E2.7"] = {
    "ti_datasheet_revision": "SBAS499C", "ti_datasheet_table": 4, "supply_V": 5, "reference_V": 4.5,
    "pga_gain": 24, "nominal_bits": 24, "documented_conditions": ti_rows,
}
assert ti_rows[1]["input_RMS_noise_uV"] / ti_rows[0]["input_RMS_noise_uV"] == 2
report = {
    "environment": {"python": platform.python_version(), "numpy": np.__version__,
                    "scipy": scipy.__version__, "matplotlib": matplotlib.__version__},
    "data_manifest_sha256": file_digest(root / "data/ch01_manifest.json"),
    "data_inputs_sha256": manifest["files_sha256"],
    "helper_sha256": file_digest(root / "src/bspsbook/measurement.py"),
    "experiments": results,
    "outputs_sha256": {str(p.relative_to(root)): file_digest(p) for p in
                       (*sorted(figures.glob("ch02_*")), *sorted(outputs.glob("ch02_*.csv")))},
}
(outputs / "ch02_run.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8",
)
print(results["E2.7"])

{'ti_datasheet_revision': 'SBAS499C', 'ti_datasheet_table': 4, 'supply_V': 5, 'reference_V': 4.5, 'pga_gain': 24, 'nominal_bits': 24, 'documented_conditions': [{'output_sps': 250, 'bandwidth_3dB_Hz': 65, 'input_RMS_noise_uV': 0.14, 'ENOB_as_reported': 19.85}, {'output_sps': 1000, 'bandwidth_3dB_Hz': 262, 'input_RMS_noise_uV': 0.28, 'ENOB_as_reported': 18.85}]}
